#### **Step 1: Import Required Libraries**


In [ ]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp)
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 3, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [ ]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/kafka_eventstream/raw_stream/bet_results"
df_raw_bet_results = spark.read.format("parquet").load(raw_transaction_path)

df_raw_bet_results = df_raw_bet_results.drop("event_id_1")
display(df_raw_bet_results.limit(10))

StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, db897192-2f71-49a7-bc39-c4d7af40f1ea)

### **Column-by-Column Transformation Plan**
user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

event_id
- Current: Clean and consistent (e.g., 68fae11e-c393-44ec-a35e-7fc6d9351629).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the bet_results dataframe for data quality checks**

In [ ]:
# Select specific columns and count nulls
null_counts = df_raw_bet_results.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['event_id', 'match_id', 'user_id', 'outcome', 'settlement_amount', 'settlement_time', 'event_date']
    ]
)

# Display results
null_counts.show()

StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 6, Finished, Available, Finished)

+--------+--------+-------+-------+-----------------+---------------+----------+
|event_id|match_id|user_id|outcome|settlement_amount|settlement_time|event_date|
+--------+--------+-------+-------+-----------------+---------------+----------+
|       0|       0|      0|      0|             1148|              0|         0|
+--------+--------+-------+-------+-----------------+---------------+----------+



#### **Step 4: Data Transformation to outcome column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels (Win, Loss, Void, Cancelled).
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid outcomes (win, loss, void, cancelled) using a canonical character signature map.
- Returned standardized labels: Win, Loss, Void, Cancelled.
- Labeled unmatched or null values as "Unknown".

In [ ]:
# 1. Pre‑compute canonical signatures

def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a word."""
    return "".join(sorted(word.lower()))

# Define the valid outcome values
canonical_values = ["Win", "Loss", "Void", "Cancelled"]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)


# 2. UDF to normalize jumbled outcomes

@udf(StringType())
def normalize_outcome(raw: str) -> str:
    if raw is None:
        return None  # Return None to allow dropping later
    signature = "".join(sorted(raw.lower().strip()))
    return sig_broadcast.value.get(signature)


# 3. Apply transformation

df_cleaned_bet_results = df_raw_bet_results.withColumn(
    "outcome",
    normalize_outcome(col("outcome"))
)


# 4. Drop rows with unrecognized/null outcome values

df_cleaned_bet_results = df_cleaned_bet_results.filter(col("outcome").isNotNull())


# 5. Inspect cleaned outcome values

df_cleaned_bet_results.select("outcome").distinct().show(truncate=False)


StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 7, Finished, Available, Finished)

+---------+
|outcome  |
+---------+
|Win      |
|Cancelled|
|Void     |
|Loss     |
+---------+



#### **Step 5: Data Transformation to match_id column**
Issue:
- Some values had inconsistent casing like MATCH_867, match_432, or mixed-case variants.

Transformation:
- Converted all match_id values to lowercase.
- Trimmed any leading or trailing whitespace.

In [ ]:
# Standardize `match_id` by converting to lowercase and trimming whitespace
df_cleaned_bet_results = df_cleaned_bet_results.withColumn(
    "match_id",
    lower(trim(col("match_id").cast("string")))
)

# Show sample records after transformation
df_cleaned_bet_results.select("match_id").show(10, truncate=False)


StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 8, Finished, Available, Finished)

+--------+
|match_id|
+--------+
|match_8 |
|match_8 |
|match_8 |
|match_8 |
|match_8 |
|match_92|
|match_8 |
|match_8 |
|match_92|
|match_92|
+--------+
only showing top 10 rows



#### **Step 6: Data Transformation to settlement_time column**
Issue:
- The column values were in ISO 8601 string format (yyyy-MM-ddTHH:mm:ssZ), not usable for date operations.

Transformation:
- Used to_timestamp() to convert string to proper timestamp format.
- Ensured time zone marker 'Z' was treated as a literal.

In [ ]:
# Convert ISO 8601 format to timestamp
df_cleaned_bet_results = df_cleaned_bet_results.withColumn(
    "settlement_time",
    to_timestamp(col("settlement_time"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Show sample converted values
df_cleaned_bet_results.select("settlement_time").show(10, truncate=False)


StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 9, Finished, Available, Finished)

+-------------------+
|settlement_time    |
+-------------------+
|2025-07-05 08:48:19|
|2025-07-05 08:40:19|
|2025-07-05 09:28:19|
|2025-07-05 08:38:19|
|2025-07-05 09:06:19|
|2025-07-05 09:15:19|
|2025-07-05 09:15:19|
|2025-07-05 09:35:19|
|2025-07-05 08:45:19|
|2025-07-05 09:13:19|
+-------------------+
only showing top 10 rows



#### **Step 7: Data Transformation to settlement_amount column**
Issue:
- Some Loss records had positive settlement amounts, which is inconsistent with expected business logic (losses should reduce value).
- Some Win records had negative settlement amounts, which should represent a gain instead.
- Records marked as Void or Cancelled had non-zero settlement amounts, which is inaccurate.
- Some values were potentially null, which needed to be handled safely.

Transformation:
- Converted Loss settlement amounts to negative if they were positive.
- Converted Win settlement amounts to positive if they were negative.
- Set all Void and Cancelled settlement amounts to 0.0.
- Retained original values for all other outcomes.

In [ ]:
# 1. Apply settlement amount corrections based on cleaned outcome values
df_cleaned_bet_results = df_cleaned_bet_results.withColumn(
    "settlement_amount",
    when(
        (col("outcome") == "Loss") & (col("settlement_amount") > 0),
        -1 * col("settlement_amount")  # Loss should be negative
    ).when(
        (col("outcome") == "Win") & (col("settlement_amount") < 0),
        abs(col("settlement_amount"))  # Win should be positive
    ).when(
        col("outcome").isin("Void", "Cancelled"),
        0.0  # Voided or Cancelled should be zero
    ).otherwise(col("settlement_amount"))  # Keep existing values
)

# 2. Remove records where settlement_amount is null
df_cleaned_bet_results = df_cleaned_bet_results.filter(col("settlement_amount").isNotNull())

# 3. Display sample of final cleaned data
df_cleaned_bet_results.select("outcome", "settlement_amount").show(20, truncate=False)


StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 10, Finished, Available, Finished)

+---------+-----------------+
|outcome  |settlement_amount|
+---------+-----------------+
|Void     |0.0              |
|Loss     |-4414.97         |
|Loss     |-3345.84         |
|Cancelled|0.0              |
|Win      |8965.61          |
|Loss     |-8423.24         |
|Win      |5633.31          |
|Win      |3833.09          |
|Cancelled|0.0              |
|Win      |817.38           |
|Win      |2990.38          |
|Win      |6287.91          |
|Loss     |-2917.91         |
|Loss     |-5479.38         |
|Void     |0.0              |
|Win      |1394.73          |
|Void     |0.0              |
|Win      |9191.55          |
|Void     |0.0              |
|Cancelled|0.0              |
+---------+-----------------+
only showing top 20 rows



#### **Step 9: Data Quality Checks**
- The script performs data quality checks on the df_cleaned_bet_results DataFrame to validate critical fields such as event_id, match_id, user_id, outcome, settlement_amount, and settlement_time. It checks for nulls, inconsistent business logic, and future timestamps, ensuring the data is suitable for downstream analytics or loading into the Silver layer. Results are aggregated and evaluated against defined thresholds to flag issues.

| Check                    | Description                                                               |
| ------------------------ | ------------------------------------------------------------------------- |
| `null_event_id`          | Should never be null                                                      |
| `null_match_id`          | Must not be null                                                          |
| `null_user_id`           | Must not be null                                                          |
| `invalid_outcome`        | Outcome must be one of: Win, Loss, Void, Cancelled                        |
| `loss_positive`          | Loss outcomes should not have positive settlement amounts                 |
| `win_negative`           | Win outcomes should not have negative settlement amounts                  |
| `void_nonzero`           | Voided bets must have settlement amount of exactly 0                      |
| `cancelled_nonzero`      | Cancelled bets must also have settlement amount of exactly 0              |
| `future_settlement_time` | Settlement time must not be in the future relative to current system time |




In [ ]:
# 1. Configuration
valid_outcomes = ["Win", "Loss", "Void", "Cancelled"]

thresholds = {
    "null_event_id":              0,
    "null_match_id":              0,
    "null_user_id":               0,
    "invalid_outcome":            0,
    "loss_positive":              0,
    "win_negative":               0,
    "void_nonzero":               0,
    "cancelled_nonzero":          0,
    "future_settlement_time":     0
}


# 2. Generate DQ Flags

dq_flags = df_cleaned_bet_results.select(
    when(col("event_id").isNull(), 1).otherwise(0).alias("null_event_id"),
    when(col("match_id").isNull(), 1).otherwise(0).alias("null_match_id"),
    when(col("user_id").isNull(),  1).otherwise(0).alias("null_user_id"),
    
    # outcome must be in valid_outcomes
    when(~col("outcome").isin(valid_outcomes), 1).otherwise(0).alias("invalid_outcome"),
    
    # settlement_amount consistency checks
    when((col("outcome") == "Loss") & (col("settlement_amount") > 0), 1).otherwise(0).alias("loss_positive"),
    when((col("outcome") == "Win")  & (col("settlement_amount") < 0), 1).otherwise(0).alias("win_negative"),
    when((col("outcome") == "Void") & (col("settlement_amount") != 0), 1).otherwise(0).alias("void_nonzero"),
    when((col("outcome") == "Cancelled") & (col("settlement_amount") != 0), 1).otherwise(0).alias("cancelled_nonzero"),
    
    # settlement_time should not be in the future
    when(col("settlement_time") > current_timestamp(), 1).otherwise(0).alias("future_settlement_time")
)


# 3. Aggregate Metrics

metrics_row = dq_flags.groupBy().sum().collect()[0]
dq_metrics = {k: metrics_row[f"sum({k})"] for k in thresholds.keys()}


# 4. Evaluate Against thresholds

dq_pass = True
print("\n  BET_RESULTS DATA QUALITY REPORT")
print("--------------------------------------------------")
for metric, threshold in thresholds.items():
    count_val = dq_metrics[metric]
    status = "✅ OK" if count_val <= threshold else "❌ FAIL"
    if count_val > threshold:
        dq_pass = False
    print(f"{metric.ljust(25)} : {str(count_val).rjust(6)}  ->  {status}")
print("--------------------------------------------------")

if dq_pass:
    print("✅  All critical DQ checks passed. Safe to load.")
else:
    print("⚠️  Data quality checks failed. Review required before loading.")


StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 11, Finished, Available, Finished)


  BET_RESULTS DATA QUALITY REPORT
--------------------------------------------------
null_event_id             :      0  ->  ✅ OK
null_match_id             :      0  ->  ✅ OK
null_user_id              :      0  ->  ✅ OK
invalid_outcome           :      0  ->  ✅ OK
loss_positive             :      0  ->  ✅ OK
win_negative              :      0  ->  ✅ OK
void_nonzero              :      0  ->  ✅ OK
cancelled_nonzero         :      0  ->  ✅ OK
future_settlement_time    :  84999  ->  ❌ FAIL
--------------------------------------------------
⚠️  Data quality checks failed. Review required before loading.


#### **Step 10: Data Load to Silver Lakehouse Layer**
This step writes the cleaned users data to the Silver layer of the Lakehouse using Delta format. It uses a MERGE strategy to enable incremental upserts—updating existing user preferences and inserting new ones—while avoiding duplicates.

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/kafka_eventstream/cleaned/cleaned_bet_results.

Prepare Columns for Consistency:
- setttlement_time: Adds the current date for use as a partition column to optimize query performance and simplify incremental loads.
- load_timestamp: Adds a consistent timestamp string (formatted as YYYYMMDDHHMMSS) that represents when the data was loaded.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key user_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by ingestion_date for efficient querying and file organization.


In [ ]:
# Define paths
silver_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_bet_results"

# Add date column for partitioning (based on event_date if available)
df_cleaned_bet_results = df_cleaned_bet_results.withColumn(
    "event_date", to_date("event_date")  
)

# Add a consistent load timestamp column (once per job run)
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_bet_results = df_cleaned_bet_results.withColumn(
    "load_timestamp", lit(load_ts)
)

# Perform MERGE if Delta table exists
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target").merge(
        df_cleaned_bet_results.alias("source"),
        "target.event_id = source.event_id"  # Replace with your unique key
    ).whenNotMatchedInsertAll().execute()

else:
    # First write: create partitioned Delta table
    df_cleaned_bet_results.write.format("delta") \
        .mode("append") \
        .partitionBy("settlement_time") \
        .save(silver_path)


StatementMeta(, cf97f194-7731-440e-8eec-21bc87fceefc, 12, Finished, Available, Finished)